## Download the hospital external dataset

In [1]:
# Download the hospital external dataset
import requests
import pandas as pd
specific_data_url = 'https://springernature.figshare.com/ndownloader/files/15591434'
output_path = '../data/landing/external_data/Hospital.csv'
specific_data_url_vic = 'https://www.healthcollect.vic.gov.au/HospitalLists/ExportList.aspx?List=MainHospitalList'
output_path_vic= '../data/landing/external_data/Hospital_vic.csv'
headers = {
     'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.36'
}
response = requests.get(specific_data_url, headers=headers)
response_vic = requests.get(specific_data_url_vic, headers=headers)
with open(output_path, 'wb') as out_file:
    out_file.write(response.content)
with open(output_path_vic, 'wb') as out_file_vic:
    out_file_vic.write(response_vic.content)

## Preprocess the dataset 

In [6]:
hospital =  pd.read_csv('../data/landing/external_data/Hospital.csv',encoding='latin1')
hospital = hospital[hospital['State'] == 'Vic']
hospital=hospital.reset_index()
hospital = hospital[['Hospital name','Postcode','Latitude','Longitude']]
#hospital.rename(columns={'original_feature_name': 'new_feature_name'}, inplace=True)
hospital['geometry'] = [[x, y] for x, y in zip(hospital['Latitude'], hospital['Longitude'])]
hospital =  hospital[['Hospital name','geometry']]

hospital['geometry'] = hospital['geometry'].apply(str)
hospital
hospital.to_csv('../data/curated/external_data/Hospital.csv')


In [3]:
hospital_vic =  pd.read_csv('../data/landing/external_data/Hospital_vic.csv', encoding='latin1')
hospital_vic = hospital_vic[['Formal Name','Postcode']] 
hospital_vic 

,Formal Name,Postcode
0,Albert Road Clinic,3205
1,Albury Wodonga Health,3690
2,"Albury Wodonga Health, Albury Campus",2640
3,Alexandra District Health,3714
4,Alfred Health,3004
...,...,...
258,Yackandandah Bush Nursing Hospital Inc.,3749
259,Yarra Ranges Health,3140
260,Yarram and District Health Service,3971
261,Yarrawonga Health,3730


In [8]:
# read the files record propertry and hospital respectively
property_data = pd.read_csv("../data/curated/merged_data/merged_data_with_facility.csv")
hospital_data = pd.read_csv("../data/curated/external_data/Hospital.csv")
hospital_data =  hospital_data[['Hospital name','geometry']]

In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
from sklearn.neighbors import BallTree
from geopy.distance import great_circle


# read the files record propertry and hospital respectively


# This function will accept a string to parse coordinate string into point object
def parse_coordinates(coord_str):
    parts = coord_str.strip('[]').split(',')
    return (float(parts[0]), float(parts[1]))


# parse coordinate string into point object for both property & hospital data
closest_hospital = []
hospital_distance = []
property_coordinates = property_data['coordinates'].apply(parse_coordinates)
all_hospital_coordinates = hospital_data['geometry'].apply(parse_coordinates)

# find the closest hospital for each property
for property_coord in property_coordinates:
    # By default: no nearest hospital and the distance is positive infinity.
    min_distance = float('inf')
    closest_hospital_geo = None
    
    for i, hospital_coord in enumerate(all_hospital_coordinates):
        # Check if the value of the coordinate point is valid
        if np.isnan(property_coord).any() or np.isnan(hospital_coord).any():
            continue
        # Calculate the distance between property and hospital, update if it is smallest
        distance = great_circle(property_coord, hospital_coord).kilometers
        if distance < min_distance:
            min_distance = distance
            closest_hospital_geo = hospital_data.loc[i, 'geometry']
    
    # Add feature to store closest hospital for the property
    closest_hospital.append(closest_hospital_geo)
    hospital_distance.append(min_distance)
property_data['closest_hospital'] = closest_hospital
property_data['hospital_distance(KM)'] = hospital_distance

In [ ]:
property_data

,name,rental_price,num_bedroom,num_bathroom,num_parking,postcode,coordinates,property_geometry,sa2_code,sa2_name,sa2_geometry,personal_income,avg_income_growth_rate(%),2021_population,avg_pop_growth_rates(%),crime_rate(%),closest_school,school_distance(KM),closest_hospital,hospital_distance(KM)
0,31 Chittagong Drive Clyde North VIC 3978,575.0,4,2,2.0,3978.0,"[-38.1053122, 145.3570863]",POINT (145.3570863 -38.1053122),212031556.0,Clyde North - South,POLYGON ((145.37034567475234 -38.0937851973191...,68495.004299,3.028540,15038.0,129.962525,0.122595,"[-38.10602, 145.37876]",1.898006,"[-38.045325, 145.347181]",6.726397
1,50 Elmtree Crescent Clyde North VIC 3978,560.0,4,2,2.0,3978.0,"[-38.0825712, 145.3561984]",POINT (145.3561984 -38.0825712),212031555.0,Clyde North - North,POLYGON ((145.3346630127421 -38.07820964472685...,68495.004299,3.028540,10652.0,29.771333,0.122595,"[-38.08468, 145.3638]",0.705427,"[-38.045325, 145.347181]",4.216162
2,7 Mortdale Lane Clyde North VIC 3978,490.0,2,2,1.0,3978.0,"[-38.0961758, 145.3800644]",POINT (145.3800644 -38.0961758),212031556.0,Clyde North - South,POLYGON ((145.37034567475234 -38.0937851973191...,68495.004299,3.028540,15038.0,129.962525,0.122595,"[-38.10602, 145.37876]",1.100561,"[-38.045325, 145.347181]",6.344909
3,54 Walhallow Drive Clyde North VIC 3978,540.0,4,2,1.0,3978.0,"[-38.1133324, 145.3457396]",POINT (145.3457396 -38.1133324),212031556.0,Clyde North - South,POLYGON ((145.37034567475234 -38.0937851973191...,68495.004299,3.028540,15038.0,129.962525,0.122595,"[-38.11488, 145.33828]",0.674921,"[-38.113312, 145.280832]",5.678594
4,10 Sicily Road Clyde North VIC 3978,520.0,4,2,2.0,3978.0,"[-38.1295789, 145.3642993]",POINT (145.3642993 -38.1295789),212031303.0,Cranbourne South,POLYGON ((145.3254642788471 -38.12742071438774...,67445.099591,2.233051,17641.0,14.250357,0.197962,"[-38.12955, 145.33886]",2.225124,"[-38.113312, 145.280832]",7.522231
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8797,61 Tongue Street Yarraville VIC 3013,630.0,2,1,0.0,3013.0,"[-37.8131463, 144.8909053]",POINT (144.8909053 -37.8131463),213031352.0,Yarraville,POLYGON ((144.85914995429522 -37.8176431203511...,92944.746058,3.965740,15651.0,0.003162,0.197223,"[-37.8137, 144.8899]",0.107655,"[-37.800508, 144.895155]",1.454065
8798,47 Mill Avenue Yarraville VIC 3013,730.0,4,3,2.0,3013.0,"[-37.8222822, 144.872198]",POINT (144.872198 -37.8222822),213031352.0,Yarraville,POLYGON ((144.85914995429522 -37.8176431203511...,92944.746058,3.965740,15651.0,0.003162,0.197223,"[-37.82104, 144.87443]",0.239821,"[-37.797407, 144.887421]",3.072331
8799,12 Adeney Street Yarraville VIC 3013,450.0,3,1,2.0,3013.0,"[-37.8163817, 144.8666543]",POINT (144.8666543 -37.8163817),213031352.0,Yarraville,POLYGON ((144.85914995429522 -37.8176431203511...,92944.746058,3.965740,15651.0,0.003162,0.197223,"[-37.8126, 144.87466]",0.819385,"[-37.797407, 144.887421]",2.789294
8800,229B Somerville Road Yarraville VIC 3013,300.0,1,1,0.0,3013.0,"[-37.8124289, 144.8779569]",POINT (144.8779569 -37.8124289),213031352.0,Yarraville,POLYGON ((144.85914995429522 -37.8176431203511...,92944.746058,3.965740,15651.0,0.003162,0.197223,"[-37.8126, 144.87466]",0.290245,"[-37.797407, 144.887421]",1.865866


In [ ]:
property_data.to_csv('../data/curated/merged_data/merged_data_with_facility.csv', index=False)